# mBART-style Encoder-Decoder: Verse → Commentary, With Attention Inspection

This notebook trains one Transformer encoder-decoder model to read a classical Tamil verse and generate its traditional commentary (*urai*), the same way a human scholar's line-by-line explanation works. It's trained in **two stages**:

1. **Denoising pretraining** — the model first learns Tamil itself by playing fill-in-the-blank on raw verse and urai text (no pairing needed).
2. **Supervised fine-tuning (SFT)** — only then does it learn the actual task: given a verse, produce its commentary.

What makes this notebook different from the plainer baseline pipeline is that the model here is instrumented for **inspection**: every forward pass also returns its internal cross-attention weights and encoder/decoder hidden states, so later cells can literally show *which verse word the model was looking at* when it generated each commentary word, and *how similar* its internal number-representations of the verse and the commentary are.

Trained on all 1,262 verse–urai pairs across Naaladiyar, Thirukadukam, and three parts of the Tholkappiyam. No train/test split — the goal here is to inspect what the model learned, not to benchmark it.

## v2 changes (corrected dataset rerun)
- **Data**: loads the corrected corpus from local `data/` (extract `classical_tamil_verse_urai_corpus.zip` there) --- **1,262 pairs, 393 Naaladiyar** (verse #398 explanation fixed). A sanity-gate cell asserts these counts before anything trains.
- **Outputs**: written to local `outputs/` instead of `/kaggle/working/`.
- Seed unchanged (3407) so results are comparable to the v1 (1,261-pair) run.
- **NEW: real 90/10 held-out split** --- denoising pretrain and SFT now train on `train_rows` only; a new held-out generation check compares output quality on unseen verses.
- **Merged:** the Thirukkural out-of-domain generalisation test (formerly notebook 04) now runs at the end of this notebook, against this same trained model --- no duplicate retraining.

In [ ]:
# (v2 local run) Kaggle boilerplate removed - numpy/pandas are imported in the next cell;
# kagglehub and ../input walking are not needed locally.
import numpy as np
import pandas as pd


In [ ]:
import json
import math
import random
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.manifold import TSNE
from tqdm.auto import tqdm

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

# --- v3: quiet, reproducible execution ---------------------------------------
import os, warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm as _tqdm
_QUIET = os.environ.get("NB_VERBOSE", "0") != "1"
def tqdm(iterable=None, *args, **kwargs):
    kwargs.setdefault("disable", _QUIET)
    kwargs.setdefault("leave", False)
    return _tqdm(iterable, *args, **kwargs)


In [ ]:
DATA_PATHS = [
    "data/naladiyar.jsonl",
    "data/sft_data-eth.jsonl",
    "data/sft_data-por.jsonl",
    "data/sft_data-sol.jsonl",
    "data/thirukadukam_tamilvu.jsonl",
]

OUT_DIR = Path("outputs/mbart_style_wordlevel_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAD = "<pad>"
UNK = "<unk>"
MASK = "<mask>"
BOS = "<bos>"
EOS = "<eos>"
VERSE_TAG = "<verse>"
URAI_TAG = "<urai>"
TASK_V2U = "<v2u>"

In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).replace("\ufeff", " ").replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

LB           = "<lb>"
TASK_DENOISE = "<denoise>"

NALADIYAR_TAG    = "<naladiyar>"
THIRUKADUKAM_TAG = "<thirukadukam>"
THOLKAPPIYAM_TAG = "<tholkappiyam>"

# v3: <lb> and the three dataset identity tags removed from the vocabulary.
# LB and the *_TAG constants remain DEFINED so that legacy display branches
# (`if tok == LB`, `SPECIAL_TOKENS - {LB}`) still resolve; they are simply
# never emitted into a token stream any more. TASK_V2U / TASK_DENOISE stay:
# they mark which objective a sequence belongs to, not which text it is from.
SPECIAL_TOKENS = [
    PAD, UNK, MASK, BOS, EOS,
    VERSE_TAG, URAI_TAG, TASK_V2U, TASK_DENOISE,
]

ANALYSIS_SKIP_TOKENS = set(SPECIAL_TOKENS)

def split_nonempty_lines(text: str):
    return [line.strip() for line in normalize_text(text).split("\n") if line.strip()]

def dataset_name_from_path(path):
    p = str(path).lower()
    if "thirukadukam" in p: return "thirukadukam"
    if "naladiyar" in p:    return "naladiyar"
    if "eth" in p:          return "tholkappiyam_eth"
    if "por" in p:          return "tholkappiyam_por"
    if "sol" in p:          return "tholkappiyam_sol"
    return Path(path).stem

def dataset_token(dataset_name):
    return {
        "naladiyar":        NALADIYAR_TAG,
        "thirukadukam":     THIRUKADUKAM_TAG,
        "tholkappiyam_eth": THOLKAPPIYAM_TAG,
        "tholkappiyam_por": THOLKAPPIYAM_TAG,
        "tholkappiyam_sol": THOLKAPPIYAM_TAG,
    }.get(dataset_name, UNK)

def verse_structure_tokens(dataset_name, verse=None):
    return []          # v3: was [dataset_token(dataset_name)]

def urai_structure_tokens(dataset_name):
    return []          # v3: was [dataset_token(dataset_name)]

def format_verse_for_model(text):
    return " ".join(split_nonempty_lines(text))   # v3: was f" {LB} ".join(...)

def format_urai_for_model(text):
    return normalize_text(text).replace("\n", " ")

def tokenize_tamil_words(text: str):
    text = normalize_text(text).lower().replace("\n", " ")   # v3: was f" {LB} "
    tokens = []
    for raw in text.split():
        if raw in ANALYSIS_SKIP_TOKENS:
            tokens.append(raw); continue
        cleaned = re.sub(r"[^\w\u0B80-\u0BFF]+", "", raw)
        if cleaned: tokens.append(cleaned)
    return tokens

def verse_model_tokens(row, task_token=None, text_override=None):
    verse_text = format_verse_for_model(text_override if text_override is not None else row["verse"])
    toks = []
    if task_token is not None: toks.append(task_token)
    toks.extend(verse_structure_tokens(row["dataset"]))
    toks.append(VERSE_TAG)
    toks.extend(tokenize_tamil_words(verse_text))
    return toks

def urai_model_tokens(row, task_token=None):
    toks = []
    if task_token is not None: toks.append(task_token)
    toks.extend(urai_structure_tokens(row["dataset"]))
    toks.append(URAI_TAG)
    toks.extend(tokenize_tamil_words(format_urai_for_model(row["urai"])))
    return toks

def load_rows_from_one(path):
    rows = []
    ds_name = dataset_name_from_path(path)
    enc = "utf-8-sig" if "naladiyar" in str(path).lower() else "utf-8"
    with open(path, "r", encoding=enc, errors="ignore") as f:
        for line in f:
            if not line.strip(): continue
            obj = json.loads(line)
            verse = normalize_text(obj.get("verse", ""))
            urai  = normalize_text(obj.get("explanation", obj.get("explaination", obj.get("urai", ""))))
            if verse and urai:
                rows.append({"dataset": ds_name, "verse": verse, "urai": urai})
    return rows

all_rows = []
for path in DATA_PATHS:
    all_rows.extend(load_rows_from_one(path))

# v2: deterministic 90/10 held-out split (seed 3407).
# Training / SFT use train_rows only; heldout_rows is never trained on.
_split_rng = random.Random(3407)
_order = list(range(len(all_rows)))
_split_rng.shuffle(_order)
_n_hold = max(1, int(0.10 * len(all_rows)))
heldout_rows = [all_rows[i] for i in _order[:_n_hold]]
train_rows   = [all_rows[i] for i in _order[_n_hold:]]
print(f"split: train={len(train_rows)}  heldout={len(heldout_rows)}")
train_pairs = [(row["verse"], row["urai"]) for row in train_rows]
pair_meta = {(row["verse"], row["urai"]): row for row in all_rows}  # v3 FIX: was train_rows
source_rows = train_rows
source_df = pd.DataFrame(source_rows)

print("rows:", len(train_rows))
print(source_df["dataset"].value_counts())
print(train_rows[0]["verse"][:120])
print(train_rows[0]["urai"][:120])

def clean_output_text(text):
    """Replace <lb> with newline and strip other special tokens for display."""
    _skip = set(SPECIAL_TOKENS) - {LB}
    parts = []
    for tok in text.split():
        if tok == LB:
            parts.append("\n")
        elif tok not in _skip:
            parts.append(tok)
    result = []
    for tok in parts:
        if tok == "\n":
            result.append("\n")
        else:
            if result and result[-1] != "\n":
                result.append(" ")
            result.append(tok)
    return "".join(result).strip()

In [ ]:
# v2 sanity gate: the corrected corpus MUST load 1,262 pairs (393 Naaladiyar).
# If this fails, the data/ folder does not contain the corrected files from
# classical_tamil_verse_urai_corpus.zip - fix that before training anything.
from collections import Counter as _Counter
_c = _Counter(r["dataset"] for r in all_rows)
print("per-dataset:", dict(_c), " total:", len(all_rows))
assert len(all_rows) == 1262, f"expected 1,262 rows, got {len(all_rows)}"
assert _c.get("naladiyar") == 393, f"expected 393 Naaladiyar rows, got {_c.get('naladiyar')}"
print("OK: corrected corpus confirmed (1,262 pairs, 393 Naaladiyar)")

In [ ]:
print("train rows:", len(train_rows))
print("train pairs:", len(train_pairs))

In [ ]:
def build_vocab(rows, min_freq=1):
    counts = Counter()

    for row in rows:
        counts.update(verse_model_tokens(row))
        counts.update(urai_model_tokens(row))

    stoi = {}
    for tok in SPECIAL_TOKENS:
        if tok not in stoi:
            stoi[tok] = len(stoi)

    for tok, freq in counts.items():
        if freq >= min_freq and tok not in stoi:
            stoi[tok] = len(stoi)

    return stoi

def encode_tokens(tokens, stoi):
    unk = stoi[UNK]
    return [stoi.get(tok, unk) for tok in tokens]

vocab = build_vocab(train_rows, min_freq=1)
itos = {v: k for k, v in vocab.items()}
print("vocab size:", len(vocab))

In [ ]:
def build_pretrain_examples(rows):
    examples = []
    for row in rows:
        examples.append({
            "kind": "verse",
            "dataset": row["dataset"],
            "text": row["verse"],
            "tokens": verse_model_tokens(row, task_token=TASK_DENOISE),
        })
        examples.append({
            "kind": "urai",
            "dataset": row["dataset"],
            "text": row["urai"],
            "tokens": urai_model_tokens(row, task_token=TASK_DENOISE),
        })
    return examples

pretrain_examples = build_pretrain_examples(train_rows)
len(pretrain_examples)

In [ ]:
class DenoiseDataset(Dataset):
    def __init__(self, examples):
        self.rows = [ex for ex in examples if len(ex["tokens"]) >= 2]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

In [ ]:
def apply_dynamic_span_mask(ids, mask_id, noise_density=0.15, max_span_len=3):
    ids = ids.copy()
    n = len(ids)
    num_to_mask = max(1, int(round(n * noise_density)))

    masked = [False] * n
    masked_count = 0

    # skip first token because it is the tag token
    valid_positions = list(range(1, n))
    if not valid_positions:
        return ids

    while masked_count < num_to_mask:
        start = random.choice(valid_positions)
        span_len = random.randint(1, max_span_len)
        for i in range(start, min(n, start + span_len)):
            if i == 0:
                continue
            if not masked[i]:
                masked[i] = True
                masked_count += 1
                if masked_count >= num_to_mask:
                    break

    return [mask_id if masked[i] else ids[i] for i in range(n)]

In [ ]:
def make_denoise_collate(vocab, noise_density=0.15, max_span_len=3):
    pad_id = vocab[PAD]
    bos_id = vocab[BOS]
    eos_id = vocab[EOS]
    mask_id = vocab[MASK]

    def collate(batch):
        src_rows, tgt_rows = [], []

        for item in batch:
            ids = encode_tokens(item["tokens"], vocab)
            corrupted = apply_dynamic_span_mask(
                ids, mask_id=mask_id, noise_density=noise_density, max_span_len=max_span_len
            )
            src = [bos_id] + corrupted + [eos_id]
            tgt = [bos_id] + ids + [eos_id]
            src_rows.append(src)
            tgt_rows.append(tgt)

        src_max = max(len(x) for x in src_rows)
        tgt_max = max(len(x) for x in tgt_rows)

        src_ids = torch.full((len(batch), src_max), pad_id, dtype=torch.long)
        src_mask = torch.zeros((len(batch), src_max), dtype=torch.bool)
        dec_in = torch.full((len(batch), tgt_max - 1), pad_id, dtype=torch.long)
        dec_tgt = torch.full((len(batch), tgt_max - 1), -100, dtype=torch.long)

        for i, (src, tgt) in enumerate(zip(src_rows, tgt_rows)):
            src_ids[i, :len(src)] = torch.tensor(src)
            src_mask[i, :len(src)] = True
            dec_in[i, :len(tgt)-1] = torch.tensor(tgt[:-1])
            dec_tgt[i, :len(tgt)-1] = torch.tensor(tgt[1:])

        return {
            "src_ids": src_ids,
            "src_mask": src_mask,
            "dec_in": dec_in,
            "dec_tgt": dec_tgt,
        }

    return collate

In [ ]:
pretrain_ds = DenoiseDataset(pretrain_examples)

pretrain_loader = DataLoader(
    pretrain_ds,
    batch_size=32,  # v3: restored (the 8 was a 4GB-local workaround) 4GB GPU (the batch x seq x 30k-vocab logits tensor OOMs at 32)
    shuffle=True,
    collate_fn=make_denoise_collate(vocab, noise_density=0.15, max_span_len=3),
)

# v2: validation loader over held-out rows (never trained on)
pretrain_val_examples = build_pretrain_examples(heldout_rows)
pretrain_val_ds = DenoiseDataset(pretrain_val_examples)
pretrain_val_loader = DataLoader(
    pretrain_val_ds,
    batch_size=32,  # v3: restored (the 8 was a 4GB-local workaround) 4GB GPU (the batch x seq x 30k-vocab logits tensor OOMs at 32)
    shuffle=False,
    collate_fn=make_denoise_collate(vocab, noise_density=0.15, max_span_len=3),
)

In [ ]:
class VerseToUraiDataset(Dataset):
    def __init__(self, rows, vocab):
        self.rows = []
        for row in rows:
            src_toks = [TASK_V2U] + verse_structure_tokens(row["dataset"], row["verse"]) + [VERSE_TAG] + tokenize_tamil_words(format_verse_for_model(row["verse"]))
            tgt_toks = [URAI_TAG] + tokenize_tamil_words(format_urai_for_model(row["urai"]))

            src_ids = encode_tokens(src_toks, vocab)
            tgt_ids = encode_tokens(tgt_toks, vocab)

            if len(src_ids) >= 2 and len(tgt_ids) >= 2:
                self.rows.append((row["dataset"], row["verse"], row["urai"], src_ids, tgt_ids))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]

In [ ]:
def make_sft_collate(vocab):
    pad_id = vocab[PAD]
    bos_id = vocab[BOS]
    eos_id = vocab[EOS]

    def collate(batch):
        src_rows, tgt_rows = [], []

        for _, _, _, src_core, tgt_core in batch:
            src = [bos_id] + src_core + [eos_id]
            tgt = [bos_id] + tgt_core + [eos_id]
            src_rows.append(src)
            tgt_rows.append(tgt)

        src_max = max(len(x) for x in src_rows)
        tgt_max = max(len(x) for x in tgt_rows)

        src_ids = torch.full((len(batch), src_max), pad_id, dtype=torch.long)
        src_mask = torch.zeros((len(batch), src_max), dtype=torch.bool)
        dec_in = torch.full((len(batch), tgt_max - 1), pad_id, dtype=torch.long)
        dec_tgt = torch.full((len(batch), tgt_max - 1), -100, dtype=torch.long)

        for i, (src, tgt) in enumerate(zip(src_rows, tgt_rows)):
            src_ids[i, :len(src)] = torch.tensor(src)
            src_mask[i, :len(src)] = True
            dec_in[i, :len(tgt)-1] = torch.tensor(tgt[:-1])
            dec_tgt[i, :len(tgt)-1] = torch.tensor(tgt[1:])

        return {
            "src_ids": src_ids,
            "src_mask": src_mask,
            "dec_in": dec_in,
            "dec_tgt": dec_tgt,
        }

    return collate

In [ ]:
sft_train_ds = VerseToUraiDataset(train_rows, vocab)

sft_train_loader = DataLoader(
    sft_train_ds,
    batch_size=32,  # v3: restored (the 8 was a 4GB-local workaround) 4GB GPU (the batch x seq x 30k-vocab logits tensor OOMs at 32)
    shuffle=True,
    collate_fn=make_sft_collate(vocab),
)

# v2: validation loader over held-out rows (never trained on)
sft_val_ds = VerseToUraiDataset(heldout_rows, vocab)
sft_val_loader = DataLoader(
    sft_val_ds,
    batch_size=32,  # v3: restored (the 8 was a 4GB-local workaround) 4GB GPU (the batch x seq x 30k-vocab logits tensor OOMs at 32)
    shuffle=False,
    collate_fn=make_sft_collate(vocab),
)

In [ ]:
def resolve_row(verse, urai=None, dataset=None):
    if dataset is not None:
        candidate = {"dataset": dataset, "verse": verse, "urai": urai if urai is not None else ""}
        return candidate
    if urai is not None and (verse, urai) in pair_meta:
        return pair_meta[(verse, urai)]
    for row in all_rows:          # v3 FIX: was train_rows, so every HELD-OUT
        # verse raised ValueError - the held-out generation cells could never
        # run. all_rows covers train + held-out.
        if row["verse"] == verse and (urai is None or row["urai"] == urai):
            return row
    # v3 FIX: do not raise. With the dataset identity tags removed,
    # verse_structure_tokens() returns [] and row["dataset"] never reaches the
    # token stream - it is display metadata only, so a miss is harmless.
    return {"dataset": "unknown", "verse": verse,
            "urai": urai if urai is not None else ""}

def build_sft_source_tokens(verse, urai=None, dataset=None):
    row = resolve_row(verse, urai=urai, dataset=dataset)
    src_tokens = [TASK_V2U] + verse_structure_tokens(row["dataset"], row["verse"]) + [VERSE_TAG] + tokenize_tamil_words(format_verse_for_model(verse))
    return row, src_tokens

def build_reconstruction_source_tokens(verse, corrupted_verse, urai=None, dataset=None):
    row = resolve_row(verse, urai=urai, dataset=dataset)
    src_tokens = [TASK_DENOISE] + verse_structure_tokens(row["dataset"], row["verse"]) + [VERSE_TAG] + tokenize_tamil_words(format_verse_for_model(corrupted_verse))
    return row, src_tokens

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
class AnalysisEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None):
        attn_out, _ = self.self_attn(
            src, src, src,
            key_padding_mask=src_key_padding_mask,
            need_weights=False,
        )
        x = self.norm1(src + self.dropout1(attn_out))
        ff = self.linear2(self.dropout(F.gelu(self.linear1(x))))
        x = self.norm2(x + self.dropout2(ff))
        return x

In [ ]:
class AnalysisDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(
        self,
        tgt,
        memory,
        tgt_mask=None,
        tgt_key_padding_mask=None,
        memory_key_padding_mask=None,
        need_weights=False,
    ):
        self_out, _ = self.self_attn(
            tgt, tgt, tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask,
            need_weights=False,
        )
        after_self = self.norm1(tgt + self.dropout1(self_out))

        cross_out, cross_weights = self.cross_attn(
            after_self, memory, memory,
            key_padding_mask=memory_key_padding_mask,
            need_weights=need_weights,
            average_attn_weights=False,
        )
        after_cross = self.norm2(after_self + self.dropout2(cross_out))

        ff = self.linear2(self.dropout(F.gelu(self.linear1(after_cross))))
        final_out = self.norm3(after_cross + self.dropout3(ff))

        return final_out, after_self, after_cross, cross_weights

In [ ]:
class MiniMBartAnalysis(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        nhead=4,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=512,
        dropout=0.1,
        max_len=512,
        pad_id=0,
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_id = pad_id

        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, dropout=dropout, max_len=max_len)

        self.encoder_layers = nn.ModuleList([
            AnalysisEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_encoder_layers)
        ])

        self.decoder_layers = nn.ModuleList([
            AnalysisDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_decoder_layers)
        ])

        self.lm_head = nn.Linear(d_model, vocab_size)

    def encode(self, src_ids, src_mask):
        x = self.embed(src_ids) * math.sqrt(self.d_model)
        x = self.pos(x)

        key_padding_mask = ~src_mask
        layer_outputs = []
        for layer in self.encoder_layers:
            x = layer(x, src_key_padding_mask=key_padding_mask)
            layer_outputs.append(x)

        return x, layer_outputs

    def decode(self, dec_in, memory, src_mask, need_weights=False):
        x = self.embed(dec_in) * math.sqrt(self.d_model)
        x = self.pos(x)

        tgt_len = dec_in.size(1)
        causal_mask = torch.triu(
            torch.full((tgt_len, tgt_len), float("-inf"), device=dec_in.device),
            diagonal=1
        )

        self_states = []
        cross_states = []
        cross_weights_all = []

        for layer in self.decoder_layers:
            x, after_self, after_cross, cross_weights = layer(
                x,
                memory,
                tgt_mask=causal_mask,
                tgt_key_padding_mask=None,
                memory_key_padding_mask=~src_mask,
                need_weights=need_weights,
            )
            self_states.append(after_self)
            cross_states.append(after_cross)
            cross_weights_all.append(cross_weights)

        logits = self.lm_head(x)
        return logits, x, self_states, cross_states, cross_weights_all

    def forward(self, src_ids, src_mask, dec_in, need_weights=False):
        memory, enc_layers = self.encode(src_ids, src_mask)
        logits, dec_out, self_states, cross_states, cross_weights_all = self.decode(
            dec_in, memory, src_mask, need_weights=need_weights
        )
        return logits, memory, enc_layers, dec_out, self_states, cross_states, cross_weights_all

In [ ]:
model = MiniMBartAnalysis(
    vocab_size=len(vocab),
    d_model=256,
    nhead=4,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=512,
    dropout=0.1,
    max_len=512,
    pad_id=vocab[PAD],
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_n = 0

    pbar = tqdm(loader)
    for batch in pbar:
        src_ids = batch["src_ids"].to(DEVICE)
        src_mask = batch["src_mask"].to(DEVICE)
        dec_in = batch["dec_in"].to(DEVICE)
        dec_tgt = batch["dec_tgt"].to(DEVICE)

        optimizer.zero_grad()
        logits, _, _, _, _, _, _ = model(src_ids, src_mask, dec_in, need_weights=False)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            dec_tgt.reshape(-1),
            ignore_index=-100,
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        n = src_ids.size(0)
        total_loss += loss.item() * n
        total_n += n
        pbar.set_postfix(loss=f"{total_loss/total_n:.4f}")

    return total_loss / total_n

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_n = 0

    for batch in loader:
        src_ids = batch["src_ids"].to(DEVICE)
        src_mask = batch["src_mask"].to(DEVICE)
        dec_in = batch["dec_in"].to(DEVICE)
        dec_tgt = batch["dec_tgt"].to(DEVICE)

        logits, _, _, _, _, _, _ = model(src_ids, src_mask, dec_in, need_weights=False)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            dec_tgt.reshape(-1),
            ignore_index=-100,
        )

        n = src_ids.size(0)
        total_loss += loss.item() * n
        total_n += n

    return total_loss / total_n

## Stage 1 — Denoising pretraining (teach it Tamil first)

Random spans of words in the raw verse/urai text are blanked out with mask tokens, and the model has to reconstruct the original — a "fill in the blank" exercise. This needs no verse-urai pairing at all; it's just learning to read and predict Tamil. 20 epochs.

**Real result from this run:** loss falls steadily from **8.94 → 4.20** over 20 epochs — the model is getting reliably better at guessing masked-out Tamil words.

In [ ]:
pretrain_history = []

for epoch in range(20):
    print(f"pretrain epoch {epoch+1}/20")
    train_loss = train_epoch(model, pretrain_loader, optimizer)
    val_loss   = eval_epoch(model, pretrain_val_loader)   # v2: track under/overfitting
    pretrain_history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})
    print(pretrain_history[-1])

In [ ]:
import random
import pandas as pd
import torch

SEED = 3407
random.seed(SEED)

# --- v3: quiet, reproducible execution ---------------------------------------
import os, warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm as _tqdm
_QUIET = os.environ.get("NB_VERBOSE", "0") != "1"
def tqdm(iterable=None, *args, **kwargs):
    kwargs.setdefault("disable", _QUIET)
    kwargs.setdefault("leave", False)
    return _tqdm(iterable, *args, **kwargs)


In [ ]:
def split_verse_lines(text):
    lines = [x.strip() for x in text.split("\n")]
    return [x for x in lines if x]

def tokenize_tamil_words_simple(text):
    return [tok for tok in text.split() if tok.strip()]

def shuffle_middle_chars(word):
    if len(word) <= 3:
        return word
    chars = list(word)
    mid = chars[1:-1]
    random.shuffle(mid)
    return chars[0] + "".join(mid) + chars[-1]

def corrupt_line_words(line, word_shuffle_prob=0.7, char_shuffle_prob=0.25):
    words = tokenize_tamil_words_simple(line)

    if len(words) > 1 and random.random() < word_shuffle_prob:
        random.shuffle(words)

    out = []
    for w in words:
        if len(w) > 3 and random.random() < char_shuffle_prob:
            out.append(shuffle_middle_chars(w))
        else:
            out.append(w)

    return " ".join(out)

def corrupt_verse(
    verse,
    line_shuffle_prob=0.8,
    word_shuffle_prob=0.7,
    char_shuffle_prob=0.25,
):
    lines = split_verse_lines(verse)

    if len(lines) > 1 and random.random() < line_shuffle_prob:
        random.shuffle(lines)

    lines = [
        corrupt_line_words(
            line,
            word_shuffle_prob=word_shuffle_prob,
            char_shuffle_prob=char_shuffle_prob,
        )
        for line in lines
    ]

    return "\n".join(lines)

In [ ]:
@torch.no_grad()
def generate_verse_reconstruction(model, vocab, itos, corrupted_verse, max_new_tokens=120):
    model.eval()

    src_tokens = [VERSE_TAG] + tokenize_tamil_words(corrupted_verse)
    src_ids_core = encode_tokens(src_tokens, vocab)
    src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]

    src_ids = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
    src_mask = torch.ones_like(src_ids, dtype=torch.bool, device=DEVICE)

    generated = [vocab[BOS], vocab[VERSE_TAG]]

    for _ in range(max_new_tokens):
        dec_in = torch.tensor([generated], dtype=torch.long, device=DEVICE)
        outputs = model(src_ids, src_mask, dec_in, need_weights=False)
        logits = outputs[0]

        next_id = int(logits[0, -1].argmax().item())
        generated.append(next_id)

        if next_id == vocab[EOS]:
            break

    out_tokens = []
    for tok_id in generated[1:]:
        tok = itos[tok_id]
        if tok == EOS:
            break
        if tok in SPECIAL_TOKENS:
            if tok == LB:
                out_tokens.append("\n")
            continue
        out_tokens.append(tok)
    result = []
    for tok in out_tokens:
        if tok == "\n":
            result.append(tok)
        else:
            if result and result[-1] != "\n":
                result.append(" ")
            result.append(tok)
    return "".join(result).strip()

## Checking the denoising actually worked

Before moving to the real task, sanity-check: take a real verse, scramble/corrupt it, and see if the model can reconstruct something close to the original. The cell below prints the **original**, **corrupted**, and **model output** side by side for a sample of verses.

In [ ]:
all_verses = [verse for verse, urai in train_pairs if verse.strip()]
probe_verses = random.sample(all_verses, min(20, len(all_verses)))

print("probe verses:", len(probe_verses))

In [ ]:
probe_rows = []

for i, verse in enumerate(probe_verses):
    corrupted = corrupt_verse(verse)
    pred = generate_verse_reconstruction(
        model=model,
        vocab=vocab,
        itos=itos,
        corrupted_verse=corrupted,
        max_new_tokens=120,
    )

    probe_rows.append({
        "id": i,
        "original_verse": verse,
        "corrupted_verse": corrupted,
        "predicted_verse": pred,
    })

probe_df = pd.DataFrame(probe_rows)
probe_df

In [ ]:
for _, row in probe_df.iterrows():
    print("=" * 90)
    print("ORIGINAL VERSE:")
    print(row["original_verse"])
    print()
    print("CORRUPTED VERSE:")
    print(row["corrupted_verse"])
    print()
    print("MODEL OUTPUT:")
    print(row["predicted_verse"])
    print()

In [ ]:
probe_rows = []

for i, verse in enumerate(probe_verses):
    corrupted = corrupt_verse(
        verse,
        line_shuffle_prob=0.6,
        word_shuffle_prob=0.5,
        char_shuffle_prob=0.10,
    )

    pred = generate_verse_reconstruction(
        model=model,
        vocab=vocab,
        itos=itos,
        corrupted_verse=corrupted,
        max_new_tokens=120,
    )

    probe_rows.append({
        "id": i,
        "original_verse": verse,
        "corrupted_verse": corrupted,
        "predicted_verse": pred,
    })

probe_df = pd.DataFrame(probe_rows)

## Stage 2 — Supervised fine-tuning (the real task)

Now the model is trained on the actual verse → urai pairs: it reads a verse and has to produce the matching commentary, word by word. 20 epochs.

**Real result from this run:** loss falls from **5.72 → 1.77** over 20 epochs. Together with the denoising numbers above, this is the same two-stage curriculum used throughout this project: learn the language first, then learn the specific task.

In [ ]:
import copy

sft_history = []
best_val, best_epoch, best_state = float("inf"), None, None

for epoch in range(20):
    print(f"sft epoch {epoch+1}/20")
    train_loss = train_epoch(model, sft_train_loader, optimizer)
    val_loss   = eval_epoch(model, sft_val_loader)        # v2: track under/overfitting

    row = {"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss}
    sft_history.append(row)
    print(row)

    # v2: epoch selection by validation loss (early stopping by checkpoint choice)
    if val_loss < best_val:
        best_val, best_epoch = val_loss, epoch + 1
        best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})

print(f"best validation epoch = {best_epoch}  (val_loss={best_val:.4f})")
model.load_state_dict(best_state)
print("restored best-val checkpoint; all downstream analysis and the saved "
      "checkpoint now use this epoch, not epoch 20")

## v2 NEW --- Epoch choice: overfitting / underfitting check

v1 trained a fixed 20+20 epochs and logged **training loss only** --- there was no way to
tell whether 20 was too few (underfitting: both curves still falling) or too many
(overfitting: validation loss rising while training loss keeps falling). The v1 symptoms
of unchecked overfitting were already visible downstream: near-perfect behaviour on
training verses next to template-collapse on out-of-domain input.

The loops above now log held-out validation loss every epoch, and SFT restores the
**best-validation** checkpoint instead of blindly keeping epoch 20. Read the curves below:

- **Underfitting**: train and val both still decreasing at epoch 20 -> train longer or raise capacity/lr.
- **Overfitting**: val bottoms out and rises while train keeps dropping -> the best epoch is the
  bottom of the val curve (annotated in the loop output); consider more dropout/weight decay.
- Record the chosen epoch in the write-up: "trained up to 20 epochs, model selected at epoch N
  by held-out loss."

In [ ]:
# Train vs validation loss curves (the under/overfitting diagnostic)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, hist, title in [(axes[0], pretrain_history, "Denoising pretrain"),
                        (axes[1], sft_history, "Verse -> Urai SFT")]:
    ep = [h["epoch"] for h in hist]
    ax.plot(ep, [h["train_loss"] for h in hist], marker="o", label="train")
    ax.plot(ep, [h["val_loss"] for h in hist], marker="s", label="val (held-out)")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "loss_curves_train_vs_val.png", dpi=150)
plt.show()

In [ ]:
torch.save(
    {
        "model_state": model.state_dict(),
        "vocab": vocab,
        "pretrain_history": pretrain_history,
        "sft_history": sft_history,
    },
    OUT_DIR / "mini_mbart_analysis.pt"
)

print("saved:", OUT_DIR / "mini_mbart_analysis.pt")

## Looking inside the model: cross-attention

While generating each commentary word, the decoder doesn't look at the whole verse equally — it assigns an **attention weight** to every verse word, effectively saying "this commentary word depends mostly on *that* verse word." This function extracts those weights, plus the model's top-5 word guesses at each position, for one example verse-urai pair.

In [ ]:
@torch.no_grad()
def inspect_cross_attention_mapping(model, vocab, itos, verse, urai, dataset=None):
    model.eval()

    row, src_tokens = build_sft_source_tokens(verse, urai=urai, dataset=dataset)
    tgt_tokens = [URAI_TAG] + tokenize_tamil_words(format_urai_for_model(urai))

    src_ids_core = encode_tokens(src_tokens, vocab)
    tgt_ids_core = encode_tokens(tgt_tokens, vocab)

    src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]
    tgt_full = [vocab[BOS]] + tgt_ids_core + [vocab[EOS]]

    src_ids = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
    src_mask = torch.ones_like(src_ids, dtype=torch.bool, device=DEVICE)
    dec_in = torch.tensor([tgt_full[:-1]], dtype=torch.long, device=DEVICE)
    dec_tgt = torch.tensor([tgt_full[1:]], dtype=torch.long, device=DEVICE)

    logits, memory, enc_layers, dec_out, self_states, cross_states, cross_weights_all = model(
        src_ids, src_mask, dec_in, need_weights=True
    )

    probs = torch.softmax(logits[0, :len(tgt_tokens)], dim=-1).cpu()
    last_cross = cross_weights_all[-1][0].cpu()
    avg_cross = last_cross.mean(dim=0)

    rows = []
    for t, tgt_word in enumerate(tgt_tokens):
        attn_row = avg_cross[t]
        max_val, max_idx = torch.max(attn_row, dim=0)

        pred_id = int(probs[t].argmax().item())
        pred_word = itos.get(pred_id, f"<id:{pred_id}>")
        gold_id = int(dec_tgt[0, t].item())
        gold_word = itos.get(gold_id, f"<id:{gold_id}>")

        topv, topi = torch.topk(probs[t], k=5)
        top5 = [itos.get(int(i), f"<id:{int(i)}>") for i in topi.tolist()]

        if int(max_idx.item()) == 0:
            attended = "<BOS>"
        elif int(max_idx.item()) == len(src_full) - 1:
            attended = "<EOS>"
        else:
            attended = src_tokens[int(max_idx.item()) - 1]

        rows.append({
            "tgt_pos": t,
            "urai_word": tgt_word,
            "gold_word": gold_word,
            "pred_word": pred_word,
            "pred_matches_gold": pred_word == gold_word,
            "most_attended_src_word": attended,
            "max_cross_attn": float(max_val.item()),
            "top5": top5,
        })

    return pd.DataFrame(rows), {
        "dataset": row["dataset"],
        "verse_text": verse,
        "urai_text": urai,
        "memory": memory.cpu(),
        "enc_layers": [x.cpu() for x in enc_layers],
        "dec_out": dec_out.cpu(),
        "self_states": [x.cpu() for x in self_states],
        "cross_states": [x.cpu() for x in cross_states],
        "cross_weights": [x.cpu() if x is not None else None for x in cross_weights_all],
        "src_tokens": src_tokens,
        "tgt_tokens": tgt_tokens,
    }

In [ ]:
probe_df, probe_cache = inspect_cross_attention_mapping(
    model=model,
    vocab=vocab,
    itos=itos,
    verse=train_pairs[0][0],
    urai=train_pairs[0][1],
)

probe_df.head(20)

## Do "verse-space" and "urai-space" numbers actually relate to each other?

Every word the encoder reads (from the verse) and every word the decoder produces (for the urai) gets its own internal vector. Here we collect a sample of those vectors and project them to 2D with t-SNE, to see visually whether encoder (verse) and decoder (urai) representations organise around shared meaning, or just cluster by which text they came from — the same question asked throughout this project's other notebooks (LSTM, BiLSTM, Attention).

In [ ]:
@torch.no_grad()
def collect_encoder_decoder_token_reps(model, rows, vocab, sample_size=100):
    model.eval()

    chosen = rows[:sample_size]
    output_rows = []

    for row in chosen:
        verse = row["verse"]
        urai = row["urai"]

        src_tokens = [TASK_V2U] + verse_structure_tokens(row["dataset"], row["verse"]) + [VERSE_TAG] + tokenize_tamil_words(format_verse_for_model(verse))
        tgt_tokens = [URAI_TAG] + tokenize_tamil_words(format_urai_for_model(urai))

        src_ids_core = encode_tokens(src_tokens, vocab)
        tgt_ids_core = encode_tokens(tgt_tokens, vocab)

        src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]
        tgt_full = [vocab[BOS]] + tgt_ids_core + [vocab[EOS]]

        src_ids = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
        src_mask = torch.ones_like(src_ids, dtype=torch.bool, device=DEVICE)
        dec_in = torch.tensor([tgt_full[:-1]], dtype=torch.long, device=DEVICE)

        logits, memory, enc_layers, dec_out, self_states, cross_states, cross_weights_all = model(
            src_ids, src_mask, dec_in, need_weights=False
        )

        enc_final = memory[0].cpu()
        dec_final = dec_out[0].cpu()

        for i, tok in enumerate(src_tokens):
            if tok in ANALYSIS_SKIP_TOKENS:
                continue
            output_rows.append({
                "dataset": row["dataset"],
                "type": "encoder_verse",
                "token": tok,
                "vector": enc_final[i + 1].numpy(),
            })

        for t, tok in enumerate(tgt_tokens):
            if tok in ANALYSIS_SKIP_TOKENS:
                continue
            output_rows.append({
                "dataset": row["dataset"],
                "type": "decoder_urai",
                "token": tok,
                "vector": dec_final[t].numpy(),
            })

    return output_rows

In [ ]:
rep_rows = collect_encoder_decoder_token_reps(
    model=model,
    rows=train_rows,
    vocab=vocab,
    sample_size=min(120, len(train_rows)),
)

len(rep_rows)

In [ ]:
freq = Counter([row["token"] for row in rep_rows])
filtered_rows = [row for row in rep_rows if freq[row["token"]] >= 3]

max_points = 1500
if len(filtered_rows) > max_points:
    filtered_rows = random.sample(filtered_rows, max_points)

X = np.stack([row["vector"] for row in filtered_rows])
labels = [row["type"] for row in filtered_rows]
tokens = [row["token"] for row in filtered_rows]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=SEED,
)

X_2d = tsne.fit_transform(X)

plot_df = pd.DataFrame({
    "x": X_2d[:, 0],
    "y": X_2d[:, 1],
    "type": labels,
    "token": tokens,
})

In [ ]:
plt.figure(figsize=(10, 8))

for ttype, color, marker in [
    ("encoder_verse", "tab:blue", "o"),
    ("decoder_urai", "tab:orange", "^"),
]:
    sub = plot_df[plot_df["type"] == ttype]
    plt.scatter(sub["x"], sub["y"], s=15, alpha=0.5, c=color, marker=marker, label=ttype)

plt.legend()
plt.title("t-SNE: Encoder Verse Reps vs Decoder Urai Reps")
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
def dataset_name_from_path(path):
    p = str(path).lower()
    if "naladiyar" in p:
        return "naladiyar"
    if "eth" in p:
        return "tholkappiyam_eth"
    if "por" in p:
        return "tholkappiyam_por"
    if "sol" in p:
        return "tholkappiyam_sol"
    return Path(path).stem

def load_pairs_with_source(paths):
    rows = []
    for path in paths:
        ds_name = dataset_name_from_path(path)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                verse = normalize_text(obj.get("verse", ""))
                urai = normalize_text(obj.get("explanation", obj.get("explaination", obj.get("urai", ""))))
                if verse and urai:
                    rows.append({
                        "dataset": ds_name,
                        "verse": verse,
                        "urai": urai,
                    })
    return rows

source_rows = load_pairs_with_source(DATA_PATHS)
source_df = pd.DataFrame(source_rows)
source_df["dataset"].value_counts()

In [ ]:
@torch.no_grad()
def collect_encoder_decoder_token_reps_with_dataset(model, source_rows, vocab, sample_size_per_dataset=50):
    model.eval()
    rows = []

    for ds_name in sorted(set(x["dataset"] for x in source_rows)):
        ds_rows = [x for x in source_rows if x["dataset"] == ds_name]
        chosen = ds_rows[:min(sample_size_per_dataset, len(ds_rows))]

        for row in chosen:
            verse = row["verse"]
            urai = row["urai"]

            src_tokens = [TASK_V2U] + verse_structure_tokens(row["dataset"], row["verse"]) + [VERSE_TAG] + tokenize_tamil_words(format_verse_for_model(verse))
            tgt_tokens = [URAI_TAG] + tokenize_tamil_words(format_urai_for_model(urai))

            src_ids_core = encode_tokens(src_tokens, vocab)
            tgt_ids_core = encode_tokens(tgt_tokens, vocab)

            src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]
            tgt_full = [vocab[BOS]] + tgt_ids_core + [vocab[EOS]]

            src_ids = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
            src_mask = torch.ones_like(src_ids, dtype=torch.bool, device=DEVICE)
            dec_in = torch.tensor([tgt_full[:-1]], dtype=torch.long, device=DEVICE)

            logits, memory, enc_layers, dec_out, self_states, cross_states, cross_weights_all = model(
                src_ids, src_mask, dec_in, need_weights=False
            )

            enc_final = memory[0].cpu()
            dec_final = dec_out[0].cpu()

            for i, tok in enumerate(src_tokens):
                if tok in ANALYSIS_SKIP_TOKENS:
                    continue
                rows.append({
                    "dataset": ds_name,
                    "type": "encoder_verse",
                    "token": tok,
                    "vector": enc_final[i + 1].numpy(),
                })

            for t, tok in enumerate(tgt_tokens):
                if tok in ANALYSIS_SKIP_TOKENS:
                    continue
                rows.append({
                    "dataset": ds_name,
                    "type": "decoder_urai",
                    "token": tok,
                    "vector": dec_final[t].numpy(),
                })

    return rows

In [ ]:
rep_rows_ds = collect_encoder_decoder_token_reps_with_dataset(
    model=model,
    source_rows=source_rows,
    vocab=vocab,
    sample_size_per_dataset=50,
)

print("raw points:", len(rep_rows_ds))

freq_ds = Counter((row["dataset"], row["token"], row["type"]) for row in rep_rows_ds)
filtered_rows_ds = [
    row for row in rep_rows_ds
    if freq_ds[(row["dataset"], row["token"], row["type"])] >= 2
]

max_points = 2000
if len(filtered_rows_ds) > max_points:
    filtered_rows_ds = random.sample(filtered_rows_ds, max_points)

print("filtered points:", len(filtered_rows_ds))

X = np.stack([row["vector"] for row in filtered_rows_ds])

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=SEED,
)

X_2d = tsne.fit_transform(X)

plot_df_ds = pd.DataFrame({
    "x": X_2d[:, 0],
    "y": X_2d[:, 1],
    "dataset": [row["dataset"] for row in filtered_rows_ds],
    "type": [row["type"] for row in filtered_rows_ds],
    "token": [row["token"] for row in filtered_rows_ds],
})

In [ ]:
dataset_colors = {
    "naladiyar": "tab:blue",
    "thirukadukam": "tab:purple",
    "tholkappiyam_eth": "tab:orange",
    "tholkappiyam_por": "tab:green",
    "tholkappiyam_sol": "tab:red",
}

type_markers = {
    "encoder_verse": "o",
    "decoder_urai": "^",
}

plt.figure(figsize=(14, 10))

for ds_name in sorted(plot_df_ds["dataset"].unique()):
    for rep_type in sorted(plot_df_ds["type"].unique()):
        sub = plot_df_ds[
            (plot_df_ds["dataset"] == ds_name) &
            (plot_df_ds["type"] == rep_type)
        ]

        plt.scatter(
            sub["x"],
            sub["y"],
            c=dataset_colors.get(ds_name, "gray"),
            marker=type_markers[rep_type],
            s=26,
            alpha=0.7,
            label=f"{ds_name} | {rep_type}",
        )

plt.title("t-SNE of Token Representations by Dataset and Representation Type\n(naladiyar=blue, thirukadukam=purple, tholkappiyam_eth=orange, por=green, sol=red)")
plt.grid(alpha=0.2)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
plot_df.to_csv(OUT_DIR / "encoder_decoder_tsne.csv", index=False)
plot_df_ds.to_csv(OUT_DIR / "encoder_decoder_tsne_by_dataset.csv", index=False)
probe_df.to_csv(OUT_DIR / "cross_attention_probe.csv", index=False)
print("saved analysis outputs to", OUT_DIR)

In [ ]:
import torch
import pandas as pd
import random

@torch.no_grad()
def generate_urai_greedy(model, vocab, itos, verse, max_new_tokens=220, urai=None, dataset=None):
    model.eval()

    row, src_tokens = build_sft_source_tokens(verse, urai=urai, dataset=dataset)
    src_ids_core = encode_tokens(src_tokens, vocab)
    src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]

    src_ids = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
    src_mask = torch.ones_like(src_ids, dtype=torch.bool, device=DEVICE)

    enc_out = model.encode(src_ids, src_mask)
    memory = enc_out[0] if isinstance(enc_out, tuple) else enc_out

    generated = [vocab[BOS], vocab[URAI_TAG]]

    for _ in range(max_new_tokens):
        dec_in = torch.tensor([generated], dtype=torch.long, device=DEVICE)
        dec_out = model.decode(dec_in, memory, src_mask, need_weights=False)
        logits = dec_out[0] if isinstance(dec_out, tuple) else dec_out
        next_id = int(logits[0, -1].argmax().item())
        generated.append(next_id)
        if next_id == vocab[EOS]:
            break

    out_tokens = []
    for tok_id in generated[1:]:
        tok = itos[tok_id]
        if tok == EOS:
            break
        if tok in SPECIAL_TOKENS:
            if tok == LB:
                out_tokens.append("\n")
            continue
        out_tokens.append(tok)
    result = []
    for tok in out_tokens:
        if tok == "\n":
            result.append(tok)
        else:
            if result and result[-1] != "\n":
                result.append(" ")
            result.append(tok)
    return "".join(result).strip()

@torch.no_grad()
def generate_verse_reconstruction(model, vocab, itos, verse, corrupted_verse, max_new_tokens=160, urai=None, dataset=None):
    model.eval()

    row, src_tokens = build_reconstruction_source_tokens(
        verse=verse,
        corrupted_verse=corrupted_verse,
        urai=urai,
        dataset=dataset,
    )
    prefix_tokens = [TASK_DENOISE] + verse_structure_tokens(row["dataset"], row["verse"]) + [VERSE_TAG]

    src_ids_core = encode_tokens(src_tokens, vocab)
    src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]
    seed = [vocab[BOS]] + encode_tokens(prefix_tokens, vocab)

    src_ids = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
    src_mask = torch.ones_like(src_ids, dtype=torch.bool, device=DEVICE)

    generated = seed[:]
    for _ in range(max_new_tokens):
        dec_in = torch.tensor([generated], dtype=torch.long, device=DEVICE)
        outputs = model(src_ids, src_mask, dec_in, need_weights=False)
        logits = outputs[0]
        next_id = int(logits[0, -1].argmax().item())
        generated.append(next_id)
        if next_id == vocab[EOS]:
            break

    out_tokens = []
    for tok_id in generated[1:]:
        tok = itos[tok_id]
        if tok in ANALYSIS_SKIP_TOKENS - {LB}:
            continue
        if tok == EOS:
            break
        out_tokens.append("\n" if tok == LB else tok)

    text = []
    for tok in out_tokens:
        if tok == "\n":
            text.append(tok)
        else:
            if text and text[-1] != "\n":
                text.append(" ")
            text.append(tok)
    return "".join(text).strip()

In [ ]:
import pandas as pd
import torch

@torch.no_grad()
def show_generation_examples_from_train_pairs(model, vocab, itos, train_pairs, n=5, max_new_tokens=220):
    model.eval()

    rows = []
    chosen = train_pairs[:n]

    for i, (verse, gold_urai) in enumerate(chosen):
        pred = generate_urai_greedy(
            model=model,
            vocab=vocab,
            itos=itos,
            verse=verse,
            max_new_tokens=max_new_tokens,
        )

        rows.append({
            "idx": i,
            "verse": verse,
            "gold_urai": gold_urai,
            "pred_urai": pred,
        })

    return pd.DataFrame(rows)

## Generation quality check: read the actual outputs

Greedy-decode a commentary for several real verses and compare against the gold (human-written) commentary side by side.

**Note on this run's outputs:** the results below are less polished than the curated examples typically presented in summaries — one prediction repeats the same phrase twice in succession, another shifts to unrelated content partway through, and at least one output in the earlier reconstruction probe returned empty. These are reported as observed, without selection for favourable results. Training outcomes on a dataset of this size vary meaningfully between runs, and this variability is itself informative about the extent and limits of what the model has learned.

In [ ]:
gen_df = show_generation_examples_from_train_pairs(
    model=model,
    vocab=vocab,
    itos=itos,
    train_pairs=train_pairs,
    n=5,
    max_new_tokens=220,
)

gen_df

In [ ]:
for _, row in gen_df.iterrows():
    print("=" * 80)
    print("VERSE:")
    print(row["verse"])
    print()
    print("GOLD URAI:")
    print(row["gold_urai"])
    print()
    print("PREDICTED URAI:")
    print(row["pred_urai"])
    print()

## v2 NEW --- Held-out generation check
The exact same greedy decode, but on the 10% of verses the model **never saw during SFT**. Compare these against the training-set generations above: the gap between them is the honest measure of memorisation vs. generalisation.

In [ ]:
gen_df_heldout = show_generation_examples_from_train_pairs(
    model=model,
    vocab=vocab,
    itos=itos,
    train_pairs=[(r["verse"], r["urai"]) for r in heldout_rows],
    n=5,
    max_new_tokens=220,
)

gen_df

In [ ]:
for _, row in gen_df_heldout.iterrows():
    print("=" * 80)
    print("VERSE:")
    print(row["verse"])
    print()
    print("GOLD URAI:")
    print(row["gold_urai"])
    print()
    print("PREDICTED URAI:")
    print(row["pred_urai"])
    print()

## The headline finding: what is the model actually attending to?

Sweeping cross-attention weight across **every** verse→urai word pair in the training data (651,748 pairs total) and keeping the top 20 by peak attention weight reveals a clear pattern.

**Real result from this run:** of the top 20 highest-attention word pairs, **19 come from Tholkappiyam Ezhuthadhikaram** and only 1 from Porulathikaram — none from Naaladiyar, Sollathikaram, or Thirukadukam at all. Reading the actual pairs (e.g. *vakaram ↔ tev*, *sāriyai ↔ ēṉaiya*, *oṟṟē ↔ aintu*), they're consistently **phonology sutra terms mapping to their definition/count words** in the commentary, "the letter class X attends to the word that defines/counts it." This makes sense: Ezhuthadhikaram is the most tightly-templated, technical-vocabulary section of the corpus, so this pattern is easiest to learn there. It also means the strongest attention signal in the entire model is concentrated in one narrow, formulaic corner of the data: not spread evenly across everything it was trained on.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

@torch.no_grad()
def collect_top_attention_pairs(model, rows, vocab, top_k=20):
    """Run all rows through the model, collect every word-pair attention score,
    return the global top_k by attention weight."""
    model.eval()
    output_rows = []

    for pair_id, row in enumerate(rows):
        verse = row["verse"]
        urai  = row["urai"]
        src_tokens = ([TASK_V2U] + verse_structure_tokens(row["dataset"], row["verse"])
                      + [VERSE_TAG] + tokenize_tamil_words(format_verse_for_model(verse)))
        tgt_tokens = [URAI_TAG] + tokenize_tamil_words(format_urai_for_model(urai))

        src_ids_core = encode_tokens(src_tokens, vocab)
        tgt_ids_core = encode_tokens(tgt_tokens, vocab)

        src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]
        tgt_full = [vocab[BOS]] + tgt_ids_core + [vocab[EOS]]

        src_ids = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
        src_mask = torch.ones_like(src_ids, dtype=torch.bool)
        dec_in   = torch.tensor([tgt_full[:-1]], dtype=torch.long, device=DEVICE)

        logits, memory, enc_layers, dec_out, self_states, cross_states, cross_weights_all = model(
            src_ids, src_mask, dec_in, need_weights=True
        )

        enc_vecs   = memory[0].cpu()
        dec_vecs   = dec_out[0].cpu()
        last_cross = cross_weights_all[-1][0].cpu()
        avg_cross  = last_cross.mean(dim=0)   # [tgt_len, src_len]

        for t, tgt_word in enumerate(tgt_tokens):
            if tgt_word in ANALYSIS_SKIP_TOKENS:
                continue
            attn_row = avg_cross[t]
            for s, src_word in enumerate(src_tokens):
                if src_word in ANALYSIS_SKIP_TOKENS:
                    continue
                output_rows.append({
                    "pair_id":    pair_id,
                    "dataset":    row["dataset"],
                    "verse_word": src_word,
                    "urai_word":  tgt_word,
                    "attn":       float(attn_row[s + 1].item()),
                    "verse_vec":  enc_vecs[s + 1].numpy(),
                    "urai_vec":   dec_vecs[t].numpy(),
                })

    attn_df = pd.DataFrame(output_rows)
    attn_df = attn_df.sort_values("attn", ascending=False).reset_index(drop=True)
    top_df  = attn_df.head(top_k).copy()
    top_df["plot_id"] = range(len(top_df))
    print(f"Total word pairs collected: {len(attn_df):,}  |  returning top {top_k}")
    print(top_df["dataset"].value_counts().to_string())
    return top_df

top_attn_df = collect_top_attention_pairs(
    model=model,
    rows=train_rows,
    vocab=vocab,
    top_k=20,
)

print("=== Top 20 cross-attention verse<->urai word pairs ===")
top_attn_df[["plot_id", "dataset", "verse_word", "urai_word", "attn"]].head(20)

## Seeing those top-20 pairs on a map

t-SNE projection of the encoder/decoder vectors for those same top-20 word pairs, with dashed lines connecting each verse word to the urai word it attends to most. Given the finding above, expect most of these connected pairs to visually cluster together, since they're almost all coming from the same narrow (Ezhuthadhikaram) part of the data.

In [ ]:
rows = []
for _, r in top_attn_df.iterrows():
    pid = int(r["plot_id"])

    rows.append({
        "plot_id": pid,
        "token": r["verse_word"],
        "side": "encoder_verse",
        "pair_label": f"{pid}: {r['verse_word']} <-> {r['urai_word']}",
        "attn": r["attn"],
        "vector": r["verse_vec"],
    })
    rows.append({
        "plot_id": pid,
        "token": r["urai_word"],
        "side": "decoder_urai",
        "pair_label": f"{pid}: {r['verse_word']} <-> {r['urai_word']}",
        "attn": r["attn"],
        "vector": r["urai_vec"],
    })

plot_rows_df = pd.DataFrame(rows)

X = np.stack(plot_rows_df["vector"].values)

tsne = TSNE(
    n_components=2,
    perplexity=min(20, max(5, len(plot_rows_df) // 5)),
    learning_rate="auto",
    init="pca",
    random_state=SEED,
)

X_2d = tsne.fit_transform(X)

plot_rows_df["x"] = X_2d[:, 0]
plot_rows_df["y"] = X_2d[:, 1]

In [ ]:
plt.figure(figsize=(22, 15))

for side, color, marker in [
    ("encoder_verse", "tab:blue", "o"),
    ("decoder_urai", "tab:orange", "^"),
]:
    sub = plot_rows_df[plot_rows_df["side"] == side]
    plt.scatter(
        sub["x"],
        sub["y"],
        c=color,
        marker=marker,
        s=55,
        alpha=0.8,
        label=side,
    )

for pid in sorted(plot_rows_df["plot_id"].unique()):
    sub = plot_rows_df[plot_rows_df["plot_id"] == pid]
    if len(sub) == 2:
        plt.plot(sub["x"], sub["y"], "k--", alpha=0.22, linewidth=0.8)
        mx = sub["x"].mean()
        my = sub["y"].mean()
        plt.text(mx, my, str(pid), fontsize=12)

plt.title("Top 20 Cross-Attention Verse-Urai Word Pairs in t-SNE")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

---
## Out-of-domain generalisation test: the Thirukural

Everything above was trained on Naaladiyar / Thirukadukam / Tholkappiyam. **The Thirukural is none of those** — a different, extremely famous classical Tamil text, written as short two-line couplets rather than the four-line quatrains or grammar-prose the model actually learned from. It is given to the model completely unseen, with no fine-tuning.

**Why deliberately test on a text it can't know:** if the output is poor, that's not a bug — it's information about exactly which of the model's assumptions are tied to the specific shape/style of its training data, versus what (if anything) about Tamil generally transferred.

In [ ]:
# -- Hardcoded Thirukural sample (out-of-domain probe) --
# Five well-known Thirukural couplets; no gold urai needed.
thirukural_rows = [
    {'dataset': 'thirukural', 'verse': 'à®…à®•à®° à®®à¯à®¤à®² à®Žà®´à¯à®¤à¯à®¤à¯†à®²à¯à®²à®¾à®®à¯ à®†à®¤à®¿ à®ªà®•à®µà®©à¯ à®®à¯à®¤à®±à¯à®±à¯‡ à®‰à®²à®•à¯', 'urai': ''},
    {'dataset': 'thirukural', 'verse': 'à®•à®±à¯à®±à®¤à®©à®¾à®²à¯ à®†à®¯ à®ªà®¯à®©à¯†à®©à¯à®•à¯‹à®²à¯ à®µà®¾à®²à®±à®¿à®µà®©à¯ à®¨à®±à¯à®±à®¾à®³à¯ à®¤à¯Šà®´à®¾à®…à®°à¯ à®Žà®©à®¿à®©à¯', 'urai': ''},
    {'dataset': 'thirukural', 'verse': 'à®‡à®©à¯à®©à®¾à®šà¯†à®¯à¯ à®¤à®¾à®°à¯ˆ à®’à®±à¯à®¤à¯à®¤à®²à¯ à®…à®µà®°à¯à®¨à®¾à®£ à®¨à®©à¯à®©à®¯à®®à¯ à®šà¯†à®¯à¯à®¤à¯ à®µà®¿à®Ÿà®²à¯', 'urai': ''},
    {'dataset': 'thirukural', 'verse': 'à®…à®©à¯à®ªà®¿à®²à®¾à®°à¯ à®Žà®²à¯à®²à®¾à®®à¯ à®¤à®®à®•à¯à®•à¯à®°à®¿à®¯à®°à¯ à®…à®©à¯à®ªà¯à®Ÿà¯ˆà®¯à®¾à®°à¯ à®Žà®©à¯à®ªà¯à®®à¯ à®‰à®°à®¿à®¯à®°à¯ à®ªà®¿à®±à®°à¯à®•à¯à®•à¯', 'urai': ''},
    {'dataset': 'thirukural', 'verse': 'à®’à®´à¯à®•à¯à®•à®®à¯ à®µà®¿à®´à¯à®ªà¯à®ªà®¨à¯ à®¤à®°à®²à®¾à®©à¯ à®’à®´à¯à®•à¯à®•à®®à¯ à®‰à®¯à®¿à®°à®¿à®©à¯à®®à¯ à®“à®®à¯à®ªà®ªà¯ à®ªà®Ÿà¯à®®à¯', 'urai': ''},
]
print(f'Thirukural verses loaded: {len(thirukural_rows)}')
for r in thirukural_rows:
    print(' -', r['verse'])

### Generating commentary for 5 real Thirukural couplets

The same greedy-decode generation process as before, now pointed at 5 well-known opening Thirukural couplets it has never seen.

In [ ]:
# ── Thirukural helper setup ───────────────────────────────────────────────────
import random as _random
_random.seed(42)

# _dataset_token_extended: maps "thirukural" → UNK (unknown dataset tag)
_orig_dataset_token = dataset_token
def _dataset_token_extended(ds):
    if ds == "thirukural": return UNK
    return _orig_dataset_token(ds)

def _verse_structure_tokens_tk(ds, verse):
    """v3: mirrors verse_structure_tokens(), which now returns []. Previously
    this fed <unk> as the dataset tag for Thirukkural, a prefix the model had
    never seen in that position during training - a confound layered on top
    of the out-of-domain test. Removed."""
    return []

SEP = "=" * 80

### Reading the actual outputs

**Real result — and this is very revealing:** all 5 predicted commentaries start with the *exact same* opening phrase — "கருத்து மேலதற்கொரு புறனடை கூறுகின்றது பொருள்" ("the meaning here relates to a supplementary matter...") — regardless of which of the 5 completely different couplets was the input. After that fixed opening, the continuations drift into vocabulary that doesn't match the Thirukural's actual subject (for instance, one prediction veers into terminology about clandestine love-affairs and go-betweens, borrowed wholesale from the Tholkappiyam's poetics sections, which has nothing to do with that particular couplet's real meaning).

**What this means in plain terms:** on completely unseen input, the model falls back on a stock formulaic opening and borrowed phrasing from its training data, rather than genuinely reasoning about the new verse in front of it. That's an honest, informative failure — it shows the model learned *a commentary-shaped template*, not a general capability to explain any Tamil verse.

In [ ]:
# ── Urai generation for Thirukural (out-of-domain generalisation test) ───────
import torch

print("=== Urai generation for Thirukural (out-of-domain generalisation test) ===")
print()

tk_gen_probe = _random.sample(thirukural_rows, min(20, len(thirukural_rows)))
gen_results  = []

@torch.no_grad()
def generate_urai_tk(model, vocab, itos, verse, max_new_tokens=200):
    """Greedy urai decode for an out-of-domain Thirukural verse.
    Builds the source tokens directly (bypassing resolve_row / build_sft_source_tokens)
    so no training-data lookup is required.
    """
    model.eval()

    src_toks = (
        [TASK_V2U]
        + _verse_structure_tokens_tk("thirukural", verse)
        + [VERSE_TAG]
        + tokenize_tamil_words(format_verse_for_model(verse))
    )
    src_ids_core = encode_tokens(src_toks, vocab)
    src_full = [vocab[BOS]] + src_ids_core + [vocab[EOS]]

    src_ids  = torch.tensor([src_full], dtype=torch.long, device=DEVICE)
    src_mask = torch.ones_like(src_ids, dtype=torch.bool, device=DEVICE)

    enc_out = model.encode(src_ids, src_mask)
    memory  = enc_out[0] if isinstance(enc_out, tuple) else enc_out

    generated = [vocab[BOS], vocab[URAI_TAG]]
    for _ in range(max_new_tokens):
        dec_in  = torch.tensor([generated], dtype=torch.long, device=DEVICE)
        dec_out = model.decode(dec_in, memory, src_mask, need_weights=False)
        logits  = dec_out[0] if isinstance(dec_out, tuple) else dec_out
        next_id = int(logits[0, -1].argmax().item())
        generated.append(next_id)
        if next_id == vocab[EOS]:
            break

    out_tokens = []
    for tok_id in generated[1:]:
        tok = itos[tok_id]
        if tok == EOS:
            break
        if tok in SPECIAL_TOKENS:
            if tok == LB:
                out_tokens.append("\n")
            continue
        out_tokens.append(tok)

    result = []
    for tok in out_tokens:
        if tok == "\n":
            result.append(tok)
        else:
            if result and result[-1] != "\n":
                result.append(" ")
            result.append(tok)
    return "".join(result).strip()


for i, row in enumerate(tk_gen_probe):
    verse = row["verse"]
    gold  = row.get("urai", "")

    pred = generate_urai_tk(model, vocab, itos, verse)
    gen_results.append({
        "verse":     verse,
        "gold_urai": gold,
        "pred_urai": pred,
    })

for i, r in enumerate(gen_results):
    print(SEP)
    print(f"THIRUKURAL #{i+1}")
    print(f"  VERSE     : {r['verse']}")
    if r["gold_urai"]:
        print(f"  GOLD URAI : {r['gold_urai'][:250]}")
    print(f"  PRED URAI : {r['pred_urai'][:250] if r['pred_urai'] else '(empty)'}")
print(SEP)
print()
print("Note: Low quality output is expected - Thirukural couplets differ structurally")
print("from the quatrain/grammar forms seen in training. This probes generalisation limits.")

### 3. Token-overlap and length analysis

Two simple numeric checks against this out-of-domain behaviour:

**Real result:** mean verse–urai token overlap is **0.000** — literally zero words shared between the generated commentary and the input couplet's own vocabulary, confirming the model isn't drawing its output from the input verse at all. Mean predicted length is **35.6 tokens** — it still produces fluent-*looking*, reasonably-sized Tamil prose; it just isn't *about* the verse it was asked to explain.

In [ ]:
# ── Token-overlap analysis: how much of the generated urai overlaps verse vocab ─
import pandas as pd

def token_overlap(verse, pred_urai):
    v_toks = set(tokenize_tamil_words(verse))
    u_toks = set(tokenize_tamil_words(pred_urai))
    if not u_toks: return 0.0
    return len(v_toks & u_toks) / len(u_toks)

rows_viz = []
for r in gen_results:
    overlap = token_overlap(r["verse"], r["pred_urai"])
    rows_viz.append({"overlap": overlap, "pred_len": len(r["pred_urai"].split()),
                     "verse": r["verse"][:60]})

if not rows_viz:
    print("No generation results to analyse.")
else:
    viz_df = pd.DataFrame(rows_viz)
    print(f"Mean verse-urai token overlap: {viz_df['overlap'].mean():.3f}")
    print(f"Mean predicted urai length   : {viz_df['pred_len'].mean():.1f} tokens")

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(viz_df["overlap"], bins=15, color="tab:blue", alpha=0.8, edgecolor="white")
    axes[0].set_xlabel("Token overlap (verse ∩ pred_urai / pred_urai)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Thirukural generalisation\nVerse–Pred token overlap")
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(viz_df["pred_len"], bins=15, color="tab:orange", alpha=0.8, edgecolor="white")
    axes[1].set_xlabel("Predicted urai length (tokens)")
    axes[1].set_title("Thirukural generalisation\nPredicted urai length distribution")
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("Out-of-domain generalisation: Thirukural", fontsize=11)
    plt.tight_layout(); plt.show()